# 04 · Case study and critical analysis

Runs the prototype pipeline for one urban location that was **not used for training** and checks the result
against an **independent** map. It then quantifies three limitations for the critical discussion:

| Section | Question |
|---|---|
| 1–3 | Prototype output: 12 monthly maps, class percentages, month-to-month and total change |
| 4 | Sanity check against the Esri 10 m Annual LULC map (Sentinel-2 based, different model, not used for training) |
| 5 | Seasonal effects: which class shares oscillate during the year without any land-use change? |
| 6 | Error inflation: how much of the calculated change is classification noise? |

The prototype app (`app/streamlit_app.py`) uses exactly the same code (`src/prototype.py`).

In [ ]:
import os, sys, json
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

import warnings
warnings.filterwarnings("ignore", message=".*restricted mode.*")   # Earth Engine quota notice

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src import config as C, gee_data as G, evaluation as E, prototype as P, plots

G.init()
print("Model:", P.MODEL_PATH)

In [ ]:
CITY = "Munich, Germany"
LAT, LON = 48.137, 11.575      # city centre; not inside any training, validation or test region
START = "2025-01"              # 12 months: Jan - Dec 2025 (test period; Esri reference available for 2025)
SIZE_PX = 1024                 # 10.24 x 10.24 km

r = P.run(LAT, LON, START, size_px=SIZE_PX)
nearest = min(C.AOIS, key=lambda n: (C.AOIS[n]["lat"] - LAT) ** 2 + (C.AOIS[n]["lon"] - LON) ** 2)
print(f"{CITY}: {r['size_km']} km square, orbit {r['orbit']}, scenes per month {list(r['scenes'].values())}")
print(f"Nearest study area: {nearest} ({C.AOIS[nearest]['lat']}, {C.AOIS[nearest]['lon']})")

## 1. Monthly land-cover maps

In [ ]:
fig = plots.maps_grid(r["maps"], r["months"], title=f"{CITY} · land cover per month (Sentinel-1 only)")
plt.show()

## 2. Class percentages per month

Share of the mapped area: class pixels / pixels with radar data × 100.

In [ ]:
plots.pct_chart(r["pct"]); plt.show()
r["pct"].rename(index=plots.month_label).round(1)

## 3. Changes in percentage points

* **Month-to-month:** share(month) − share(previous month)
* **Total:** share(last month) − share(first month)

In [ ]:
display(r["month_to_month"].iloc[1:].rename(index=plots.month_label).round(2))
plots.total_change_chart(r["total"]); plt.show()
r["total"].round(2).to_frame("total change (pp)")

## 4. Sanity check against an independent map

The **Esri 10 m Annual LULC** map is derived from Sentinel-2 by a different model (Impact Observatory) and was
**not used for training** (training labels came from Dynamic World). It has the same eight classes. Because it is
annual, it is compared with the most frequent class per pixel over the 12 months.

In [ ]:
pred_annual = P.annual_majority(r["maps"], r["months"])
plots.reference_pair(pred_annual, r["ref_map"], r["ref_year"], r["months"]); plt.show()
print(f"Esri year: {r['ref_year']} | pixel agreement: {100 * r['ref_agreement']:.1f} %")
r["ref_table"].round(1)

In [ ]:
cm = E.confusion(r["ref_map"], pred_annual)       # rows: Esri, columns: model
with np.errstate(invalid="ignore"):
    cm_norm = cm / cm.sum(1, keepdims=True)
present = cm.sum(1) > 0
fig, ax = plt.subplots(figsize=(7.5, 6))
ax.imshow(np.where(present[:, None], cm_norm, np.nan), cmap="Blues", vmin=0, vmax=1)
ax.set_xticks(range(len(C.CLASSES)), C.CLASSES, rotation=45, ha="right")
ax.set_yticks(range(len(C.CLASSES)), C.CLASSES)
for i in np.where(present)[0]:
    for j in range(len(C.CLASSES)):
        ax.text(j, i, f"{cm_norm[i, j]:.2f}", ha="center", va="center", fontsize=8,
                color="white" if cm_norm[i, j] > 0.5 else "black")
ax.set_xlabel("model (annual majority)"); ax.set_ylabel(f"Esri {r['ref_year']}")
ax.set_title(f"{CITY}: agreement with Esri, row-normalised")
plt.tight_layout(); plt.show()
E.scores(cm)["per_class"].round(3)

## 5. Seasonal effects

Within one year, the true land use of a city changes by at most a few tenths of a percent. Larger swings in a
class share are therefore mostly **seasonal backscatter changes** (vegetation growth and harvest, soil moisture,
snow, flooding) that the model turns into class changes.

In [ ]:
seasonal = pd.DataFrame({"mean_%": r["pct"].mean(), "min_%": r["pct"].min(), "max_%": r["pct"].max(),
                         "range_pp": r["pct"].max() - r["pct"].min(), "std_pp": r["pct"].std()})
seasonal["month_of_max"] = r["pct"].idxmax().map(lambda m: plots.month_label(m) if isinstance(m, str) else m)
display(seasonal.sort_values("range_pp", ascending=False).round(2))

fig, ax = plt.subplots(figsize=(11, 4))
for c, color in zip(C.CLASSES, C.CLASS_COLORS):
    if r["pct"][c].max() > 1:
        ax.plot(range(12), r["pct"][c], marker="o", color=color, label=c, lw=2)
ax.set_xticks(range(12), [plots.month_label(m) for m in r["months"]], rotation=45, ha="right")
ax.set_ylabel("% of area"); ax.grid(alpha=0.3); ax.legend(loc="center left", bbox_to_anchor=(1.01, 0.5))
ax.set_title("Class shares through the year (classes above 1 %)")
plt.tight_layout(); plt.show()

In [ ]:
fig = plots.s1_grid(r["s1"], r["months"], band=1)
fig.suptitle("VH backscatter per month: seasonal changes in the radar signal itself", y=1.01)
plt.show()
monthly_db = pd.DataFrame({m: np.nanmean(a, (0, 1)) for m, a in r["s1"].items() if a is not None},
                          index=C.S1_BANDS).T.rename(index=plots.month_label)
monthly_db.round(2)

## 6. How classification errors inflate change

**Gross vs. net change.** The net change of a class share hides how many pixels actually switched class: pixels
switching *into* and *out of* a class cancel out. The share of pixels whose class differs from the previous month
(gross change) is an upper bound for real change plus noise.

In [ ]:
def gross_change(maps, months):
    rows = {}
    for prev, cur in zip(months[:-1], months[1:]):
        a, b = maps[prev], maps[cur]
        if a is None or b is None:
            continue
        keep = (a != C.IGNORE) & (b != C.IGNORE)
        rows[plots.month_label(cur)] = 100 * (a[keep] != b[keep]).mean()
    return pd.Series(rows, name="pixels changing class (%)")

smoothed = P.temporal_mode(r["maps"], r["months"])
pct_s = P.class_percentages(smoothed, r["months"])
mtm_s, total_s = P.changes(pct_s)

gross = pd.DataFrame({"raw": gross_change(r["maps"], r["months"]),
                      "3-month majority": gross_change(smoothed, r["months"])})
display(gross.round(1))
print(f"Mean share of pixels changing class per month: raw {gross['raw'].mean():.1f} %, "
      f"smoothed {gross['3-month majority'].mean():.1f} %")

In [ ]:
compare = pd.DataFrame({
    "mean |month-to-month| raw (pp)": r["month_to_month"].abs().mean(),
    "mean |month-to-month| smoothed (pp)": mtm_s.abs().mean(),
    "total raw (pp)": r["total"],
    "total smoothed (pp)": total_s,
})
compare.round(2)

**Expected spurious change.** If a pixel is classified correctly with probability *p* each month and errors in
different months are independent, a pixel whose land cover does not change still appears to change between two
months with probability ≈ 1 − p² (both months wrong in the same way is rare). With the overall accuracy from the
test set, this gives the share of pixels that would appear to change even if nothing changed:

In [ ]:
results = json.loads((ROOT / "weights" / "results.json").read_text()) if (ROOT / "weights" / "results.json").exists() else None
p_test = results["test"][results["selected_model"]]["OA"] if results else None
for label, p in [("test-set OA", p_test), ("agreement with Esri here", r["ref_agreement"])]:
    if p is not None:
        print(f"p = {p:.3f} ({label}): expected share of pixels with spurious change between two months "
              f"≈ {100 * (1 - p ** 2):.1f} %")
print(f"Observed share of pixels changing class per month: {gross['raw'].mean():.1f} %")

### Summary for the critical discussion

*(Written after the run.)*

* **Agreement with the independent map:** …
* **Classes most affected by seasonal effects:** …
* **Apparent vs. real change:** …
* **Effect of temporal smoothing:** …